# Many Smaller GPUs

- TODO: expand bullets

- laying out a possibly large model across many smaller GPUs
- a model that can also be used to test custom allocation strategies
- objective: [graph](./gpus.pdf)

In [1]:
import numpy as np
import tensorflow as tf
from datetime import datetime

- more preps

In [2]:
ks = tf.keras
kl = ks.layers
cfg = tf.config.experimental

- partition physical GPUs into custom-sized virtual GPUs
- components, layers, of the model can then expect properly allocated virtual GPU ids
- given the parameter-driven "resource" requirements of our layers, we can develop heuristics for partitioning and allocating the physical devices

In [3]:
devs = ((None, ), (1000, 1000, 1000, 1000, 1000, 1000), (1000, 1000, 1000, 1000, 1000, 1000))
cfg.set_visible_devices(cfg.get_visible_devices('CPU')[:1], 'CPU')
cfg.set_visible_devices(cfg.get_visible_devices('GPU')[:len(devs) - 1], 'GPU')
for d, ms in zip(cfg.get_visible_devices(), devs):
    vs = [cfg.VirtualDeviceConfiguration(m) for m in ms]
    cfg.set_virtual_device_configuration(d, vs)
devs = cfg.list_logical_devices('CPU')
devs += cfg.list_logical_devices('GPU')
print('devices:', [d.name for d in devs])

devices: ['/job:localhost/replica:0/task:0/device:CPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:3', '/job:localhost/replica:0/task:0/device:GPU:4', '/job:localhost/replica:0/task:0/device:GPU:5', '/job:localhost/replica:0/task:0/device:GPU:6', '/job:localhost/replica:0/task:0/device:GPU:7', '/job:localhost/replica:0/task:0/device:GPU:8', '/job:localhost/replica:0/task:0/device:GPU:9', '/job:localhost/replica:0/task:0/device:GPU:10', '/job:localhost/replica:0/task:0/device:GPU:11']


- turn off "soft" allocation for now

In [4]:
tf.config.set_soft_device_placement(False)

- a most basic, custom "dense" layer as a component in our configurable stack
- we aim to "lay" this stack onto our virtual GPUs as a functional forward-backward pipeline that fits in our combined GPU-space
- each layer would therefore use a predetermined virtual GPU

In [5]:
class Layer(kl.Layer):
    def __init__(self, i, ps, **kw):
        super().__init__(**kw)
        self.idx = min(i + 1, len(devs) - 1)
        self.ps = ps

    def build(self, input_shape):
        s = input_shape[-1]
        with tf.device(devs[self.idx].name):
            self.w = self.add_weight(name='l_w', shape=(s, s))
            self.b = self.add_weight(name='l_b', shape=(s, ))
        return super().build(input_shape)

    def call(self, x):
        with tf.device(devs[self.idx].name):
            y = tf.einsum('bi,ij->bj', x, self.w) + self.b
        return y

- a basic "sequential" Keras model is all what we need
- once the input (and the output too) is shaped, we chain our "dense" layers in the middle
- Keras' `summary` feature is vary handy to confirm our model is laid out as intended

In [6]:
def model_for(ps):
    m = ks.Sequential()
    m.add(kl.Dense(ps.dim_hidden, input_dim=ps.dim_input, name='in'))
    for i in range(ps.num_layers):
        m.add(Layer(i, ps, name=f'lay_{i}'))
    m.add(kl.Dense(ps.dim_input, name='out'))
    m.compile(optimizer=ps.optimizer(), loss=ps.loss(), metrics=[ps.metrics()])
    print(m.summary())
    return m

- before we can run our model, we need to establish our parameters
- a simple Python `dict` works the best to keep things organized, unique and sorted out

In [7]:
params = dict(
    dim_hidden=1000,
    dim_input=100,
    loss=ks.losses.MeanAbsoluteError,
    metrics=ks.metrics.MeanAbsoluteError,
    num_layers=10,
    optimizer=ks.optimizers.SGD,
)

- the drawback of string keyed `dict`s is just that, the strings can be misspelled and hundreds of potentially misnamed parameters cause unneeded pain and suffering
- Python's automatically verified `attributes` come to the rescue: a simple, straightforward and functional `Params` class

In [8]:
class Params:
    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)

- let's create our `Params` instance and a handy training data set (with testing and verification all built in)

In [10]:
ps = Params(**params)
import numpy as np
d = np.ones((100, ps.dim_input))

- finally we are ready to compile our model
- the summary shows that, just as expected, it has over 10 million weights randomly picked
- we then bring them "inline" through millions of multiplications and additions by our many virtual GPUs, only to verify that our input `ones` are in fact just a series of `1`s

In [11]:
m = model_for(ps)

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
in (Dense)                   (None, 1000)              101000    
_________________________________________________________________
lay_0 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_1 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_2 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_3 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_4 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_5 (Layer)                (None, 1000)              1

- running the model gives us the familiar Keras output showing a nice convergence of a trivial problem across easily configurable GPUs

In [12]:
from datetime import datetime
ld = datetime.now().strftime('%Y%m%d-%H%M%S')
ld = f'/tmp/logs/{ld}'
cs = [ks.callbacks.TensorBoard(log_dir=ld, histogram_freq=1)]
m.fit(d, d, callbacks=cs, epochs=10, batch_size=10)

Train on 100 samples
Epoch 1/10
100/100 [==============================] - 2s 17ms/sample - loss: 0.4796 - mean_absolute_error: 0.4796
Epoch 2/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.2872 - mean_absolute_error: 0.2872
Epoch 3/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.2414 - mean_absolute_error: 0.2414
Epoch 4/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.2252 - mean_absolute_error: 0.2252
Epoch 5/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.1988 - mean_absolute_error: 0.1988
Epoch 6/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.1984 - mean_absolute_error: 0.1984
Epoch 7/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.1734 - mean_absolute_error: 0.1734
Epoch 8/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.1551 - mean_absolute_error: 0.1551
Epoch 9/10
100/100 [==============================] - 1s 7ms/sample - loss

- and now let's fire up TensorBoard and visually confirm that our stack of "dense" layers is connected just as expected
- if you haven't run the code, an already generated graph is [here](./gpus.pdf)

In [1]:
%load_ext tensorboard

In [2]:
%tensorboard --logdir /tmp/q/logs